In [1]:
import pandas as pd
import numpy as np

---

## Level 1 — Airport weather

`weather` logs wind speed and visibility at irregular intervals. `flights` records departures at their own times. Attach the most recent weather reading to each flight using `merge_asof`.

1. Sort both by `time` and merge.
2. Add `delay_score = wind_speed + (100 - visibility) / 10`. Which flight had the highest delay score? Use `np.argmax`.
3. Grouped by `airline`, compute mean `delay_score` and mean `wind_speed` using named agg.
4. Use `np.argsort` on the mean delay scores to rank airlines worst → best. Print the ranked airline names.

In [16]:
weather = pd.DataFrame({
    'time':       pd.to_datetime(['06:00','07:15','08:30','09:00','10:20','11:45'], format='%H:%M'),
    'wind_speed': [12, 18, 25, 22, 15, 10],
    'visibility': [80, 60, 40, 45, 70, 90],
})

flights = pd.DataFrame({
    'time':    pd.to_datetime(['06:45','07:50','08:15','09:30','10:05','11:00'], format='%H:%M'),
    'flight':  ['AA101', 'UA202', 'DL303', 'SW404', 'BA505', 'UA606'],
    'airline': ['AA', 'UA', 'DL', 'SW', 'BA', 'UA'],
})

# Your code here

weather = weather.sort_values('time')
flights = flights.sort_values('time')
m = pd.merge_asof(
    flights, 
    weather,
    on = 'time'

)

m['delay_score'] = m['wind_speed'] +(100-m['visibility'])/10

g = m.groupby('airline').agg(
    mean_delay_score = ('delay_score','mean'),
    mean_wind_speed = ('wind_speed','mean')
)


print('from worst to best:',g.index[np.argsort(-g['mean_delay_score'])])

from worst to best: Index(['BA', 'SW', 'DL', 'UA', 'AA'], dtype='object', name='airline')


---

## Level 2 — Factory sensor readings

**New concept: `tolerance` in `merge_asof`**

By default `merge_asof` matches any past value no matter how old. The `tolerance` parameter adds a maximum allowed gap — if the nearest match is further away than the threshold, NaN is returned instead.

```python
pd.merge_asof(left, right, on='time', tolerance=pd.Timedelta('5min'))
```

Useful when a stale reading is worse than no reading — a temperature from an hour ago says nothing about current conditions.

`sensor` logs temperature every 10 minutes. `events` records equipment alerts at irregular times. Attach each event's temperature **only if a sensor reading exists within 5 minutes**.

1. Sort both by `time`. Merge with `tolerance=pd.Timedelta('5min')`.
2. How many events have no sensor reading (NaN temperature)? Print their event IDs.
3. Use `np.nanmean` to find the average temperature across all events.
4. For events that *did* get a reading, use `np.percentile` to find the 75th percentile temperature.

In [27]:
sensor = pd.DataFrame({
    'time':        pd.to_datetime(['09:00','09:10','09:20','09:30','09:40','09:50','10:00'], format='%H:%M'),
    'temperature': [72.1, 74.3, 78.9, 82.4, 81.0, 76.5, 75.2],
})

events = pd.DataFrame({
    'time':     pd.to_datetime(['09:03','09:18','09:32','09:47','09:54','10:06'], format='%H:%M'),
    'event_id': ['E1', 'E2', 'E3', 'E4', 'E5', 'E6'],
    'severity': ['low', 'high', 'medium', 'low', 'high', 'medium'],
})

# Your code here

sensor = sensor.sort_values('time')
events = events.sort_values('time')

m = pd.merge_asof(
    events, 
    sensor, 
    on = 'time',
    tolerance=pd.Timedelta('5min')
)

lna = [ei for ei, t in zip(m['event_id'],m['temperature']) if pd.isna(t)]
print(len(lna), 'events have no temperature data:', lna)


print('average temperature is:', np.nanmean(m['temperature']))
t = m.dropna(subset='temperature')['temperature']
print('75th percentile is: ',np.percentile(t, 75))


3 events have no temperature data: ['E2', 'E4', 'E6']
average temperature is: 77.0
75th percentile is:  79.45


---

## Level 3 — Currency pair alignment

`eurusd` and `gbpusd` record exchange rates against the dollar, but on alternating months — neither series has a complete record.

Build a full monthly view and analyze the relationship between the two rates.

1. Use `merge_ordered` with `fill_method='ffill'`. Drop any rows still NaN after the fill.
2. Add `spread = eurusd - gbpusd`. Which month had the widest spread (most negative)? Which had the narrowest?
3. Use `np.corrcoef` on the two rate columns. Do the currencies move together?
4. Use a list comprehension to collect months where `eurusd > 1.090`.

In [42]:
eurusd = pd.DataFrame({
    'date':   pd.to_datetime(['2023-01','2023-03','2023-05','2023-07','2023-09','2023-11'], format='%Y-%m'),
    'eurusd': [1.085, 1.076, 1.092, 1.105, 1.058, 1.088],
})

gbpusd = pd.DataFrame({
    'date':   pd.to_datetime(['2023-02','2023-04','2023-06','2023-08','2023-10','2023-12'], format='%Y-%m'),
    'gbpusd': [1.234, 1.248, 1.271, 1.283, 1.215, 1.269],
})

# Your code here

m = pd.merge_ordered(eurusd,
                     gbpusd,
                     on = 'date',
                     fill_method='ffill').dropna()
m['spread'] = m['eurusd'] - m['gbpusd']

print(m['spread'].idxmin(),'has the widest spread')
print(m['spread'].idxmax(), 'has the narrowest')

c = np.corrcoef(m['eurusd'],m['gbpusd'])[0,1]
print(f'correlation is {c:.2f}, mildly correlated')

[mo for mo in m['date'] if m.loc[m['date']==mo,'eurusd'].values[0] >1.090]

8 has the widest spread
10 has the narrowest
correlation is 0.35, mildly correlated


[Timestamp('2023-05-01 00:00:00'),
 Timestamp('2023-06-01 00:00:00'),
 Timestamp('2023-07-01 00:00:00'),
 Timestamp('2023-08-01 00:00:00')]